# Phụ lục: Bằng chứng và tái lập

## Bộ bằng chứng được sử dụng

Báo cáo chọn audit toàn dải mới của TV1/TV2 và giữ audit TV3/TV4/TV5 đã đối chiếu. Cumulative của cả năm thành viên được nhập nguyên byte, kiểm phạm vi, định danh và hash. Tệp TV1 hợp lệ về cấu trúc nhưng không có record, nên chỉ ghi nhận bằng chứng và không gộp các số corpus của tệp này. Snapshot cũ và benchmark trước được giữ để truy vết, không cộng trùng cùng run ID.

ZIP/JSON bổ sung do thành viên gửi trực tiếp; pipeline log và bản sao dữ liệu riêng không được công bố. Kiểm hash bảo đảm đúng tệp đã nhập, chưa thay việc kiểm định độc lập corpus trên Drive.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '_config.yml').exists())
sys.path.insert(0, str(ROOT / 'utils'))
from preliminary_results import (load_members, progress_rows, assignment_outcomes,
    quality_rows, error_groups, performance_rows, run_range, session_members, coverage_rows, corpus_metrics, snapshot_rows, percentage, paper_timings)
members = load_members(ROOT)
sessions = session_members(ROOT, members)
def table(rows):
    frame = pd.DataFrame(rows, dtype=object)
    frame = frame.where(frame.notna(), 'Chưa đủ bằng chứng')
    def number(value):
        if isinstance(value, float):
            return f'{value:,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
        if isinstance(value, int) and not isinstance(value, bool):
            return f'{value:,}'.replace(',', '.')
        return str(value)
    display(HTML(frame.map(number).to_html(index=False, escape=True, border=0, classes='dataframe')))


In [ ]:
for s in sessions:
    p=s['provenance']
    source='['+p['archive_name']+']('+p['drive_url']+')' if p.get('drive_url') else p['archive_name']+' (gửi trực tiếp)'
    display(Markdown(s['session']+': '+source+'; SHA-256 ZIP: `'+p['archive_sha256']+'`'))
for m in members:
    for item in (m['provenance'] or {}).get('supplementary_archives',[]):
        display(Markdown(m['member']+': '+item['archive_name']+'; SHA-256: `'+item['sha256']+'`'))
    for item in (m['provenance'] or {}).get('supplementary_files',[]):
        display(Markdown(m['member']+': '+item['filename']+'; SHA-256: `'+item['sha256']+'`'))
    if m['cumulative']:
        receipt=json.loads((m['cumulative_directory']/'provenance.json').read_text(encoding='utf-8'))
        display(Markdown(m['member']+' cumulative — '+receipt['original_filename']+'; SHA-256: `'+receipt['sha256']+'`'))
table(snapshot_rows(members))

## Phạm vi và nhất quán audit

Năm audit được đối chiếu với dải cá nhân. TV1 có inputs và audit trùng nội dung trong ZIP; TV2/TV5 liệt kê đủ định danh. TV2 mới ghi false khi còn paper fail, thay cờ true mâu thuẫn trước đó. Các audit đều false vì validation còn lỗi; điều này không phủ nhận 25.000 lookup thành công.

Riêng TV2, known_version_count ghi 3.887 nhưng cộng versions trong paper_results của chính audit được 7.987, cũng bằng cumulative. Cờ hoàn tất đã sửa nhưng trường tổng version cần bên xuất kiểm tra lại. Bảng sau giữ nguyên số gốc để truy vết; thống kê source dùng 7.987 từ cumulative.

In [ ]:
table(coverage_rows(members))
table(assignment_outcomes(members))

## Môi trường từng phiên đo

Xác nhận CPU-only là thông tin notebook ghi lại. Dòng nhắc kiểm tra CPU-only của TV2 chưa phải xác nhận đã thực hiện. Run resume không được đổi thành benchmark fresh bằng cách đổi nhãn. Với ZIP TV1 mới, run ID được đọc từ đối số --run-id đã ghi vì benchmark không có trường run_id cấp gốc.

In [ ]:
table([{'Phiên':s['session'],'Run ID':s['benchmark']['run_id'],
        'Bắt đầu (UTC)':s['benchmark']['started_at'],'Kết thúc (UTC)':s['benchmark']['finished_at'],
        'Source commit':s['benchmark']['environment'].get('source_commit'),
        'Colab':s['benchmark']['environment'].get('is_colab'),
        'CPU logic':s['benchmark']['environment'].get('logical_cpu_count'),
        'CPU-only ghi nhận':s['benchmark']['environment'].get('cpu_only_confirmation')} for s in sessions])

## Tái lập

Cài môi trường theo README, chạy kiểm thử rồi build Jupyter Book. Bảng, biểu đồ và CSV timing tính lại từ artifacts đã commit, không gọi API crawl khi build. Code tính toán được ẩn khỏi HTML.

[MS1_Cumulative_Statistics.ipynb](https://colab.research.google.com/github/HoangLongNha/arxiv-project-report/blob/main/colab/MS1_Cumulative_Statistics.ipynb) xuất thống kê corpus hiện có. Dừng tiến trình ghi, chọn đúng thành viên và data. Export đọc checkpoint, sidecar và tệp thực tế, kiểm tra thay đổi trong lúc đọc; không tải lại paper hoặc sửa corpus. Validation trong cumulative là báo cáo đã lưu, không phải audit mới.

| Tệp | Phạm vi |
|---|---|
| assignment-audit.json | Coverage và kiểm định trên dải cá nhân |
| cumulative-statistics.json | Toàn dải từ dữ liệu tích lũy tại một thời điểm |
| statistics.json | Paper được manifest riêng lượt chạy chọn |
| inputs.json | Dải dự định audit; không thay kết quả thực thi |
| benchmark/samples/timings | Thời gian và tài nguyên một phiên |
| provenance.json | Nguồn và SHA-256 của bằng chứng |

## Dữ liệu còn cần bổ sung

TV1 cần xuất lại cumulative từ đúng data đã audit; bản đã gửi ghi 0 record và 5.000 unseen dù audit trước xác nhận đủ dải. Không cần suy rằng phải crawl lại: trước hết đối chiếu DATA và checkpoint. TV5 đã bổ sung bản sau sửa lookup ngày 10/11. Muốn kết luận benchmark thu thập ban đầu, cần lịch sử đo đầy đủ hoặc phép đo mới với biên đo và môi trường Colab CPU-only được xác nhận.